# Lab dl_03 · An MLP language model with embeddings (Bengio et al. 2003)

**Goal.** Predict the next character from the previous `block_size` characters using learned **embeddings** and an MLP,
train with mini-batches, tune the learning rate, and measure generalisation on held-out words.

### Theory recap
Each character $c$ gets a vector $C[c]\in\mathbb R^{e}$ (a row of the embedding table, i.e. $\mathrm{onehot}(c)^\top C$).
For a context $(c_{t-3}, c_{t-2}, c_{t-1})$:
$$\mathbf x = [C[c_{t-3}];\,C[c_{t-2}];\,C[c_{t-1}]]\in\mathbb R^{3e},\quad \mathbf h = \tanh(W_1\mathbf x + \mathbf b_1),\quad
\text{logits} = W_2\mathbf h + \mathbf b_2.$$
Parameters: $Ve + (\text{block}\cdot e)\,h + h + hV + V$. Characters used in similar contexts get similar embeddings, which is
how the model generalises to contexts it never saw — the core idea of
[Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf).

**Learning-rate range test:** take one model, do one SGD step at each learning rate of an exponentially increasing sequence
$10^{-3}\ldots10^{1}$, and plot loss vs. $\log_{10}$ lr; pick an lr a bit below where the loss is lowest / starts to explode.

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

BOS = "."

def load_words(path="data/tinyshakespeare.txt", min_len=2):
    text = Path(path).read_text(encoding="utf-8").lower()
    return sorted({w for w in re.findall(r"[a-z]+", text) if len(w) >= min_len})

def build_vocab(words):
    chars = sorted(set("".join(words)))
    stoi = {BOS: 0, **{c: i + 1 for i, c in enumerate(chars)}}
    return stoi, {i: c for c, i in stoi.items()}

words = load_words()
stoi, itos = build_vocab(words)
V = len(stoi)

## 1. Dataset and split
`build_dataset(words, stoi, block_size)` → `X` (N, block_size) and `Y` (N,). Contexts start as all zeros (`.`); for `emma`, block 3:
`[.,.,.]→e, [.,.,e]→m, [.,e,m]→m, [e,m,m]→a, [m,m,a]→.`

`split_words(words, seed=42)` shuffles a copy with `random.Random(seed)` and splits 80/10/10 into train/val/test.

In [ ]:
def build_dataset(words, stoi, block_size):
    # TODO
    raise NotImplementedError


def split_words(words, seed=42, frac=(0.8, 0.1)):
    # TODO
    raise NotImplementedError

In [ ]:
block_size = 3
tr, va, te = split_words(words)
Xtr, Ytr = build_dataset(tr, stoi, block_size)
Xva, Yva = build_dataset(va, stoi, block_size)
Xte, Yte = build_dataset(te, stoi, block_size)
print(Xtr.shape, Xva.shape, Xte.shape)

## 2. The model
Use exactly these attribute names: `self.emb = nn.Embedding(V, n_embd)`, `self.fc1 = nn.Linear(block_size * n_embd, n_hidden)`, `self.fc2 = nn.Linear(n_hidden, V)`, and keep `self.block_size`. `forward(idx)` maps (B, block_size) → logits (B, V) with a tanh hidden layer.

In [ ]:
class MLPLM(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=10, n_hidden=200):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, idx):
        # TODO
        raise NotImplementedError


@torch.no_grad()
def evaluate(model, X, Y):
    # TODO: cross-entropy over the whole set, returned as a float (use model.eval() / model.train())
    raise NotImplementedError

## 3. Training and the learning-rate range test
`train(model, X, Y, steps, lr, batch_size, seed, lr_decay_at=None)`: mini-batch SGD (`torch.optim.SGD`), batches drawn with `torch.randint(..., generator=torch.Generator().manual_seed(seed))`; if `lr_decay_at` is given divide lr by 10 from that step. Returns per-step losses.

`lr_range_test(model, X, Y, lrs, batch_size, seed)`: one SGD step per lr (in the given order) on the same model; returns the loss **before** each step.

In [ ]:
def train(model, X, Y, steps=1000, lr=0.1, batch_size=32, seed=0, lr_decay_at=None):
    # TODO
    raise NotImplementedError


def lr_range_test(model, X, Y, lrs, batch_size=32, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
lrs = (10 ** torch.linspace(-3, 1, 400)).tolist()
rt = lr_range_test(MLPLM(V, block_size, 10, 200), Xtr, Ytr, lrs)
plt.plot(torch.linspace(-3, 1, 400), rt); plt.xlabel("log10 lr"); plt.ylabel("loss"); plt.ylim(1.5, 6); plt.show()

In [ ]:
torch.manual_seed(0)
model = MLPLM(V, block_size, n_embd=10, n_hidden=200)
print("params:", sum(p.numel() for p in model.parameters()))
losses = train(model, Xtr, Ytr, steps=30000, lr=0.1, batch_size=64, lr_decay_at=20000)
plt.plot(torch.tensor(losses).view(-1, 100).mean(1)); plt.ylabel("train loss (avg of 100)"); plt.show()
print(f"train {evaluate(model, Xtr, Ytr):.4f}  val {evaluate(model, Xva, Yva):.4f}   (bigram val NLL was ~2.4-2.5)")

## 4. Look inside: embeddings and samples
Train a model with `n_embd=2` to plot the embedding table directly. Do vowels cluster?

In [ ]:
torch.manual_seed(0)
m2 = MLPLM(V, block_size, n_embd=2, n_hidden=100)
train(m2, Xtr, Ytr, steps=15000, lr=0.1, batch_size=64, lr_decay_at=10000)
E = m2.emb.weight.detach()
plt.figure(figsize=(6, 6)); plt.scatter(E[:, 0], E[:, 1], s=200, c="lightblue")
for i in range(V):
    plt.text(E[i, 0], E[i, 1], itos[i], ha="center", va="center")
plt.title("2-D character embeddings"); plt.show()

@torch.no_grad()
def sample(model, n=10, seed=0):
    g = torch.Generator().manual_seed(seed); out = []
    for _ in range(n):
        ctx, w = [0] * model.block_size, []
        while len(w) < 30:
            ix = torch.multinomial(F.softmax(model(torch.tensor([ctx])), 1), 1, generator=g).item()
            if ix == 0: break
            w.append(itos[ix]); ctx = ctx[1:] + [ix]
        out.append("".join(w))
    return out
print(sample(model))

### Reflect (on paper)
1. Count the parameters of your model by hand and compare. Which layer dominates?
2. Your val loss is higher than train loss. How large is the gap, and what would you change (size, regularisation, data) to shrink it?
3. Increase `block_size` to 5 or 8. Does the val loss improve? Why does an MLP scale poorly with context length (think about `fc1`)?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_03_mlp_lm", globals())